## Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from ngboost import NGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer

## Load the Data

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
data = pd.read_csv(url, names=columns)
data.head()

In [ ]:
data.to_csv("Cleveland Heart Disease Dataset.csv",index=False) # Save data to local machine

## Data Preprocessing

In [ ]:
data = pd.read_csv("Cleveland Heart Disease Dataset.csv")
print(len(data))
data.head()

In [ ]:
data.info()

In [ ]:
# Simple code to display distinct values and their count for each column in the DataFrame
for col in data.columns:
    unique_values = data[col].unique()
    unique_count = len(unique_values)
    print(f"Column: {col}")
    print(f"Distinct Values: {unique_values}")
    print(f"Count of Unique Values: {unique_count}\n")

### 1. Handeling records with invalid data

In [ ]:
# Count occurrences of '?' in 'ca' and 'thal' column
count_question_mark = data[['ca', 'thal']].apply(lambda col: (col == '?').sum())
print(count_question_mark) # Display the result

In [ ]:
data.replace('?', np.nan, inplace=True)  # Replace "?" with NaN
data.dropna(inplace=True)
print(len(data))
data.head()

### 2. Convert Target Variable into Binary variable

In [ ]:
data['target'] = data['target'].apply(lambda x: 1 if x > 0 else 0)
print(len(data))
data.head()

### 3. Splitting features and target

In [ ]:
# Splitting features and target
X = data.drop(columns='target')
y = data['target']

### 4. Feature Scaling for Numerical attributes

In [ ]:
from sklearn.preprocessing import StandardScaler

# Define the numerical columns you want to normalize
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Initialize the scaler
scaler = StandardScaler()

# Apply z-score normalization only to the numerical columns
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Print the transformed data
print(len(X))
X.head()

### 5. One hot Encoding for Categorical Variable

In [ ]:
# Apply One-Hot Encoding to all categorical columns
X = pd.get_dummies(X, columns=['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal'])
print(len(X))
X.head()

### 4. Splitting the Data into Train and Test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Ensembles Methods

### 1. Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,            # Number of trees
    max_depth=10,                # Maximum depth of each tree
    min_samples_split=5,         # Minimum samples to split a node
    min_samples_leaf=2,          # Minimum samples in a leaf node
    max_features='sqrt',         # Number of features to consider for each split
    bootstrap=True,              # Use bootstrapping
    random_state=42              # Set seed for reproducibility
)

In [ ]:
# Train the model
rf_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
y_pred

In [ ]:
y_pred_proba

In [ ]:
# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("accuracy :",accuracy)
print("Precission :",precision)
print("Recall :",recall)
print("f1-Score :",f1)

### 2. Extra Random Tree

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, f1_score

In [ ]:
# Define Extra Trees model
extra_trees_model = ExtraTreesClassifier(
    n_estimators=100,        # Number of trees in the forest
    max_depth=None,          # Maximum depth of each tree; None allows full depth
    min_samples_split=2,     # Minimum samples required to split a node
    min_samples_leaf=1,      # Minimum samples required in a leaf node
    #max_features= ,     # Number of features to consider for best split
    bootstrap=False,         # No replacement sampling by default
    random_state=42          # Seed for reproducibility
)

In [ ]:
# Train the model
extra_trees_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_etr = extra_trees_model.predict(X_test)
y_pred_proba_etr = extra_trees_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate model performance
accuracy_etr = accuracy_score(y_test, y_pred_etr)
auc_roc_etr = roc_auc_score(y_test, y_pred_proba_etr)
precision_etr = precision_score(y_test, y_pred_etr)
recall_etr = recall_score(y_test, y_pred_etr)
f1_etr = f1_score(y_test, y_pred_etr)

print(f"Extra Trees performance metric:" )
print("accuracy :",accuracy_etr)
print("Precission :",precision_etr)
print("Recall :",recall_etr)
print("f1-Score :",f1_etr)
print("AUC-Score :",auc_roc_etr)

### 3. XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,               # Number of boosting rounds
    learning_rate=0.1,              # Step size shrinkage
    max_depth=6,                    # Maximum tree depth
    subsample=0.8,                  # Fraction of samples used per tree
    colsample_bytree=0.8,           # Fraction of features used per tree
    gamma=1,                        # Minimum loss reduction to make a split
    use_label_encoder=False,        # Disable the use of the label encoder for warnings

     eval_metric='logloss',          # Evaluation metric for binary classification
    random_state=42                 # Set seed for reproducibility
)

In [ ]:
# Train the model
xgb_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate model performance
accuracy_xgb = accuracy_score(y_test, y_pred)
auc_roc_xgb = roc_auc_score(y_test, y_pred_proba)
precision_xgb = precision_score(y_test, y_pred)
recall_xgb = recall_score(y_test, y_pred)
f1_xgb = f1_score(y_test, y_pred)

print(f"XGBoost performance metric:" )
print("accuracy :",accuracy_xgb)
print("Precission :",precision_xgb)
print("Recall :",recall_xgb)
print("f1-Score :",f1_xgb)
print("AUC-Score :",auc_roc_xgb)

### 4. NGBoost

In [ ]:
from ngboost.distns import Bernoulli  # Distribution for binary classification
from sklearn.tree import DecisionTreeRegressor
from ngboost.scores import LogScore  # Scoring function

In [ ]:
ngb_model = NGBClassifier(
    Dist=Bernoulli,                     # Distribution for binary classification
    Score=LogScore,                      # Scoring function
    Base=DecisionTreeRegressor(max_depth=3),  # Base learner
    n_estimators=100,                    # Number of boosting iterations
    learning_rate=0.01,                  # Learning rate
    minibatch_frac=0.8,                  # Fraction of samples per boosting round
    natural_gradient=True,               # Use natural gradients
    random_state=42                      # Set seed for reproducibility
)


In [ ]:
# Train the model
ngb_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred = ngb_model.predict(X_test)
y_pred_proba =ngb_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate model performance
accuracy_ngb = accuracy_score(y_test, y_pred)
auc_roc_ngb = roc_auc_score(y_test, y_pred_proba)
precision_ngb = precision_score(y_test, y_pred)
recall_ngb = recall_score(y_test, y_pred)
f1_ngb = f1_score(y_test, y_pred)

print(f"XGBoost performance metric:" )
print("accuracy :",accuracy_ngb)
print("Precission :",precision_ngb)
print("Recall :",recall_ngb)
print("f1-Score :",f1_ngb)
print("AUC-Score :",auc_roc_ngb)

### 5. Adaboost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, f1_score

In [ ]:
# Define AdaBoost model with a decision tree as the base estimator
ada_model = AdaBoostClassifier(
    #base_estimator=DecisionTreeClassifier(max_depth=3),  # Base learner
    n_estimators=100,                                    # Number of boosting iterations
    learning_rate=0.01,                                  # Learning rate
    random_state=42                                      # Set seed for reproducibility
)

In [ ]:
# Train the model
ada_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_ada = ada_model.predict(X_test)
y_pred_proba_ada = ada_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate model performance
accuracy_adb = accuracy_score(y_test, y_pred_ada)
auc_roc_adb = roc_auc_score(y_test, y_pred_proba_ada)
precision_adb = precision_score(y_test, y_pred_ada)
recall_adb = precision_score(y_test, y_pred_ada)
f1_adb = f1_score(y_test, y_pred_ada)


print(f"AdaBoost performance metric:" )
print("accuracy :",accuracy_adb)
print("Precission :",precision_adb)
print("Recall :",recall_adb)
print("f1-Score :",f1_adb)
print("AUC-Score :",auc_roc_adb)

### 6. LightGBM

In [ ]:
#!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, f1_score

In [ ]:
# Define LightGBM model
lgbm_model = LGBMClassifier(
    n_estimators=100,              # Number of boosting iterations
    learning_rate=0.01,            # Learning rate
    subsample=0.8,                 # Fraction of samples per boosting round
    colsample_bytree=0.8,          # Fraction of features per boosting round
    random_state=42                # Set seed for reproducibility
)

In [ ]:
# Train the model
lgbm_model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_lgbm = lgbm_model.predict(X_test)
y_pred_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate model performance
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
auc_roc_lgbm = roc_auc_score(y_test, y_pred_proba_lgbm)
precision_lgbm = precision_score(y_test, y_pred_lgbm)
recall_lgbm = recall_score(y_test, y_pred_lgbm)
f1_lgbm = f1_score(y_test, y_pred_lgbm)


print(f"LightGBM performance metric:" )
print("accuracy :",accuracy_lgbm)
print("Precission :",precision_lgbm)
print("Recall :",recall_lgbm)
print("f1-Score :",f1_lgbm)
print("AUC-Score :",auc_roc_lgbm)

### Comparative analysis

In [ ]:
model_name_list = ['Random Forest','Extra Random Tree','XGBoost','NGBoost','Adaboost','LightGBM']
models_list = [rf_model, extra_trees_model,xgb_model,ngb_model,ada_model,lgbm_model]

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

metrics_list = []

for model_name, model in zip(model_name_list, models_list):
    y_pred = model.predict(X_test)
    
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_pred)
    
    metrics_list.append({
        'Model': model_name,
        'Precision': precision,
        'Recall': recall,
        'Accuracy': accuracy,
        'F1 Score': f1,
        'AUC Score': roc,
        
    })

metrics_df = pd.DataFrame(metrics_list)

metrics_df